In [ ]:
%pip install "pip<24.1"
%pip install "torch==2.5.1" "torchaudio==2.5.1"
%pip install git+https://github.com/liyaodev/fairseq.git
%pip install faiss-cpu ffmpeg-python loguru praat-parselmouth pyworld torchcrepe edge-tts nest_asyncio av librosa scipy soundfile
%pip install rvc-python --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 89.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 66.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s et

In [2]:
from google.colab import drive
import os
from rvc_python.infer import RVCInference

drive.mount('/content/drive')
MODEL_PATH = "/content/drive/MyDrive/mochigo/anya.pth"

print("\nBooting up PyTorch on NVIDIA GPU...")
rvc = RVCInference(device="cuda:0")
rvc.load_model(MODEL_PATH)
rvc.set_params(
    f0up_key=10,       
    f0method="rmvpe", 
    index_rate=0,
    filter_radius=3,
    resample_sr=0,
    rms_mix_rate=0.25,
    protect=0.33
)
print("The mochi has awaken!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Booting up PyTorch on NVIDIA GPU...
is_half:True, device:cuda:0


/usr/local/lib/python3.12/dist-packages/rvc_python/modules/vc/modules.py:103: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.cpt = torch.load(sid, map_location="cpu")
/u

gin_channels: 256 self.spk_embed_dim: 109
Model anya.pth loaded.
The mochi has awaken!


In [5]:
import asyncio
import edge_tts
from rvc_python.infer import RVCInference
import os
import re
import soundfile as sf
import numpy as np
from google.colab import drive
import nest_asyncio
from IPython.display import Audio, display, clear_output


nest_asyncio.apply()

drive.mount('/content/drive')

MODEL_PATH = "/content/drive/MyDrive/mochigo/anya.pth"
INDEX_PATH = "/content/drive/MyDrive/mochigo/anya.index"

print("\nBooting up System on NVIDIA GPU. Please hold...")

rvc = RVCInference(device="cuda:0")
rvc.load_model(MODEL_PATH)
rvc.set_params(
    f0up_key=10,
    f0method="rmvpe",   
    index_rate=0,       
    filter_radius=3,
    resample_sr=0,
    rms_mix_rate=0.25,
    protect=0.33
)

async def warmup():
    print("Pre-loading features into VRAM...")
    await edge_tts.Communicate("a", "en-US-AriaNeural").save("warm.mp3")
    rvc.infer_file("warm.mp3", "warm.wav")

asyncio.run(warmup())
# -------------------------------

def split_by_language(text):
    jp_pattern = r'([\u3000-\u303f\u3040-\u309f\u30a0-\u30ff\uff00-\uff9f\u4e00-\u9faf\u3400-\u4dbf]+)'
    raw_chunks = re.split(jp_pattern, text)
    parsed_chunks = []
    for chunk in raw_chunks:
        if not chunk.strip(): continue
        parsed_chunks.append(('ja' if re.search(jp_pattern, chunk) else 'en', chunk.strip()))
    return parsed_chunks

def cleanup():
    for f in os.listdir('/content'):
        if f.startswith("temp_") or f.startswith("warm") or f == "final_mochi.wav":
            try: os.remove(f)
            except: pass

if __name__ == "__main__":
    cleanup()
    clear_output() # Clears the messy boot logs so you start with a clean screen!
    print("READY. Mochi is listening...\n")
    print("="*40)
    
    # --- THE INTERACTIVE LOOP ---
    while True:
        user_input = input("\nWhat should Mochi say? (or type 'exit'): ")
        
        if user_input.lower() == 'exit':
            print("\nShutting down AI...")
            break
            
        if not user_input.strip():
            continue
            
        print("Processing on GPU...")
        chunks = split_by_language(user_input)
        combined_audio = []
        target_sr = None
        
        try:
            for i, (lang, content) in enumerate(chunks):
                base_audio = f"temp_base_{i}.mp3"
                final_audio = f"temp_anya_{i}.wav"
                
                # 1. Route to Native Speakers
                if lang == 'en':
                    asyncio.run(edge_tts.Communicate(content, "en-US-AriaNeural").save(base_audio))
                else:
                    asyncio.run(edge_tts.Communicate(content, "ja-JP-NanamiNeural").save(base_audio))
                    
                # 2. Apply Anime Filter
                rvc.infer_file(base_audio, final_audio)
                
                # 3. Read the audio data
                data, sr = sf.read(final_audio)
                combined_audio.append(data)
                target_sr = sr
            
            if combined_audio:
                # 4. Stitch chunks into one file
                final_mix = np.concatenate(combined_audio)
                sf.write("final_mochi.wav", final_mix, target_sr)
                
                # 5. Clear the screen and pop up the new audio player
                clear_output(wait=True)
                print("="*40)
                print(f"Mochi: {user_input}")
                display(Audio("final_mochi.wav", autoplay=True))
                
        except Exception as e:
            print(f"\n[ERROR] {e}")
            
        cleanup()

Mochi: Hi I am mochi your english study companion. 楽しく英語を学びましょう



Shutting down AI...
